In [11]:
!mlrun config set -a http://172.24.54.148:30070

> 2024-07-12 07:01:01,472 [warning] Client version with higher version than server version isn't supported, align your client to the server version: {'parsed_server_version': Version(major=1, minor=0, patch=5, prerelease=None, build=None), 'parsed_client_version': Version(major=1, minor=6, patch=4, prerelease=None, build=None)}
Updating configuration in .env file C:\Users\sushant/.mlrun.env


In [12]:
import mlrun
# # Use local service
#mlrun.set_environment("http://localhost:8080", artifact_path="./")
# # Use remote service
# mlrun.set_environment("<remote-service-url>", access_key="xyz", username="joe")

In [13]:
project = mlrun.get_or_create_project("tutorial", "./", user_project=True)

> 2024-07-12 07:01:34,538 [warning] Error during request handling, retrying: {'exc': "HTTPConnectionPool(host='localhost', port=8080): Max retries exceeded with url: /api/v1/projects/tutorial-sushant (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x0000023D8D6F5700>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))", 'retry_count': 0, 'url': 'http://localhost:8080/api/v1/projects/tutorial-sushant', 'method': 'GET'}
> 2024-07-12 07:01:57,901 [warning] Error during request handling, retrying: {'exc': "HTTPConnectionPool(host='localhost', port=8080): Max retries exceeded with url: /api/v1/projects/tutorial-sushant (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x0000023D8D7374F0>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))", 'retry_count': 1, 'url': 'http://loc

MLRunRuntimeError: HTTPConnectionPool(host='localhost', port=8080): Max retries exceeded with url: /api/v1/projects/tutorial-sushant (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x0000023D8D6FBFA0>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it')): Failed retrieving project tutorial-sushant

In [ ]:
%%writefile data-prep.py

import pandas as pd
from sklearn.datasets import load_breast_cancer

import mlrun


@mlrun.handler(outputs=["dataset", "label_column"])
def breast_cancer_generator():
    """
    A function which generates the breast cancer dataset
    """
    breast_cancer = load_breast_cancer()
    breast_cancer_dataset = pd.DataFrame(
        data=breast_cancer.data, columns=breast_cancer.feature_names
    )
    breast_cancer_labels = pd.DataFrame(data=breast_cancer.target, columns=["label"])
    breast_cancer_dataset = pd.concat(
        [breast_cancer_dataset, breast_cancer_labels], axis=1
    )

    return breast_cancer_dataset, "label"

Writing data-prep.py


In [ ]:
data_gen_fn = project.set_function(
    "data-prep.py",
    name="data-prep",
    kind="job",
    image="mlrun/mlrun",
    handler="breast_cancer_generator",
)
project.save() 

In [14]:
project.name
gen_data_run = project.run_function("data-prep", handler="main", local=True)
gen_data_run.wait_for_completion()

> 2024-07-12 07:08:20,518 [warning] Error during request handling, retrying: {'exc': "HTTPConnectionPool(host='localhost', port=8080): Max retries exceeded with url: /api/v1/projects/tutorial-sushant (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x0000023D8D7E29D0>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))", 'retry_count': 0, 'url': 'http://localhost:8080/api/v1/projects/tutorial-sushant', 'method': 'GET'}
> 2024-07-12 07:08:43,880 [warning] Error during request handling, retrying: {'exc': "HTTPConnectionPool(host='localhost', port=8080): Max retries exceeded with url: /api/v1/projects/tutorial-sushant (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x0000023D8D8A83D0>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))", 'retry_count': 1, 'url': 'http://loc